In [ ]:
# ============================================================
# NOTEBOOK 10 — STRUCTURED OBJECT + CAPTION PREPARATION
# CELL 1 — ENVIRONMENT
# ============================================================

import os
import json
import random
import numpy as np
import pandas as pd

from pathlib import Path

print("=" * 70)
print("NOTEBOOK 10 — STRUCTURED OBJECT + CAPTION PREPARATION")
print("=" * 70)


# ------------------------------------------------------------
# MOUNT DRIVE
# ------------------------------------------------------------

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False
)


# ------------------------------------------------------------
# PROJECT PATH
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Image_captioning"
)


assert PROJECT_ROOT.exists(), (
    f"Project directory not found:\n{PROJECT_ROOT}"
)


print("\nProject:")
print(PROJECT_ROOT)

print("\n✓ ENVIRONMENT READY")

NOTEBOOK 10 — STRUCTURED OBJECT + CAPTION PREPARATION
Mounted at /content/drive

Project:
/content/drive/MyDrive/Image_captioning

✓ ENVIRONMENT READY


In [ ]:
# ============================================================
# CELL 2 — PATH CONFIGURATION
# ============================================================

print("=" * 70)
print("PATH CONFIGURATION")
print("=" * 70)


# ------------------------------------------------------------
# INPUT FILES
# ------------------------------------------------------------

CAPTIONS_FILE = (
    PROJECT_ROOT
    / "Dataset"
    / "Processed"
    / "Captions"
    / "captions_processed.csv"
)


RTDETR_FILE = (
    PROJECT_ROOT
    / "Outputs"
    / "RTDETR_Detection"
    / "Final"
    / "rtdetr_x_detections_full.csv"
)


# ------------------------------------------------------------
# OUTPUT DIRECTORY
# ------------------------------------------------------------

OUTPUT_DIR = (
    PROJECT_ROOT
    / "Dataset"
    / "Processed"
    / "Structured_Object_Features"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

print("\nCaptions:")
print(CAPTIONS_FILE)
print("Exists:", CAPTIONS_FILE.exists())


print("\nRT-DETR-X detections:")
print(RTDETR_FILE)
print("Exists:", RTDETR_FILE.exists())


print("\nOutput directory:")
print(OUTPUT_DIR)


# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

assert CAPTIONS_FILE.exists(), (
    "Processed captions file not found."
)

assert RTDETR_FILE.exists(), (
    "RT-DETR-X detection file not found."
)


print("\n✓ ALL REQUIRED INPUT FILES FOUND")

PATH CONFIGURATION

Captions:
/content/drive/MyDrive/Image_captioning/Dataset/Processed/Captions/captions_processed.csv
Exists: True

RT-DETR-X detections:
/content/drive/MyDrive/Image_captioning/Outputs/RTDETR_Detection/Final/rtdetr_x_detections_full.csv
Exists: True

Output directory:
/content/drive/MyDrive/Image_captioning/Dataset/Processed/Structured_Object_Features

✓ ALL REQUIRED INPUT FILES FOUND


In [ ]:
# ============================================================
# CELL 3 — LOAD CAPTIONS
# ============================================================

print("=" * 70)
print("LOADING PROCESSED CAPTIONS")
print("=" * 70)


captions_df = pd.read_csv(
    CAPTIONS_FILE
)


print(
    "\nShape:",
    captions_df.shape
)


print(
    "\nColumns:"
)

print(
    list(captions_df.columns)
)


# ------------------------------------------------------------
# REQUIRED COLUMNS
# ------------------------------------------------------------

required_caption_columns = [
    "image_id",
    "dataset",
    "image_name",
    "caption",
    "processed_caption"
]


for column in required_caption_columns:

    assert column in captions_df.columns, (
        f"Missing caption column: {column}"
    )


# ------------------------------------------------------------
# BASIC CHECKS
# ------------------------------------------------------------

assert (
    captions_df["processed_caption"]
    .notna()
    .all()
), "Missing processed captions found."


print(
    "\nTotal caption records:",
    len(captions_df)
)


print(
    "Unique images:",
    captions_df["image_id"].nunique()
)


print(
    "\n✓ CAPTIONS VERIFIED"
)

LOADING PROCESSED CAPTIONS

Shape: (199369, 7)

Columns:
['image_id', 'dataset', 'image_name', 'caption', 'processed_caption', 'model_caption', 'caption_length']

Total caption records: 199369
Unique images: 39874

✓ CAPTIONS VERIFIED


In [ ]:
# ============================================================
# CELL 4 — LOAD RT-DETR-X DETECTIONS
# ============================================================

print("=" * 70)
print("LOADING RT-DETR-X DETECTIONS")
print("=" * 70)


detections_df = pd.read_csv(
    RTDETR_FILE
)


print(
    "\nShape:",
    detections_df.shape
)


print(
    "\nColumns:"
)

print(
    list(detections_df.columns)
)


# ------------------------------------------------------------
# REQUIRED COLUMNS
# ------------------------------------------------------------

required_detection_columns = [

    "image_id",

    "image_path",

    "object_name",

    "confidence",

    "xmin",

    "ymin",

    "xmax",

    "ymax"
]


for column in required_detection_columns:

    assert column in detections_df.columns, (
        f"Missing detection column: {column}"
    )


print(
    "\nDetection records:",
    len(detections_df)
)


print(
    "Unique images:",
    detections_df["image_id"].nunique()
)


print(
    "Unique objects:",
    detections_df["object_name"].nunique()
)


print(
    "\n✓ RT-DETR-X DATA LOADED"
)

LOADING RT-DETR-X DETECTIONS

Shape: (496803, 9)

Columns:
['image_id', 'image_path', 'object_name', 'confidence', 'xmin', 'ymin', 'xmax', 'ymax', 'objects_detected']

Detection records: 496803
Unique images: 39874
Unique objects: 80

✓ RT-DETR-X DATA LOADED


In [ ]:
# ============================================================
# CELL 5 — CLEAN DETECTION RECORDS
# ============================================================

print("=" * 70)
print("CLEANING DETECTION RECORDS")
print("=" * 70)


before_count = len(
    detections_df
)


clean_df = detections_df.copy()


# ------------------------------------------------------------
# REMOVE MISSING OBJECTS
# ------------------------------------------------------------

clean_df = clean_df[
    clean_df["object_name"].notna()
].copy()


# ------------------------------------------------------------
# CLEAN OBJECT NAMES
# ------------------------------------------------------------

clean_df["object_name"] = (
    clean_df["object_name"]
    .astype(str)
    .str.strip()
)


# ------------------------------------------------------------
# REMOVE EMPTY PLACEHOLDER OBJECTS
# ------------------------------------------------------------

invalid_objects = {
    "",
    "nan",
    "NaN",
    "None",
    "none",
    "null",
    "NULL"
}


clean_df = clean_df[
    ~clean_df["object_name"].isin(
        invalid_objects
    )
].copy()


# ------------------------------------------------------------
# NUMERIC CONVERSION
# ------------------------------------------------------------

numeric_columns = [

    "confidence",
    "xmin",
    "ymin",
    "xmax",
    "ymax"
]


for column in numeric_columns:

    clean_df[column] = pd.to_numeric(
        clean_df[column],
        errors="coerce"
    )


# ------------------------------------------------------------
# REMOVE INVALID NUMERIC RECORDS
# ------------------------------------------------------------

clean_df = clean_df.dropna(
    subset=numeric_columns
).copy()


# ------------------------------------------------------------
# REMOVE INVALID BOXES
# ------------------------------------------------------------

clean_df = clean_df[
    (clean_df["xmax"] > clean_df["xmin"])
    &
    (clean_df["ymax"] > clean_df["ymin"])
].copy()


# ------------------------------------------------------------
# CONFIDENCE RANGE
# ------------------------------------------------------------

clean_df = clean_df[
    (clean_df["confidence"] >= 0.0)
    &
    (clean_df["confidence"] <= 1.0)
].copy()


after_count = len(
    clean_df
)


print(
    "\nOriginal records:",
    before_count
)


print(
    "Valid detection records:",
    after_count
)


print(
    "Removed records:",
    before_count - after_count
)


print(
    "\nImages with valid detections:",
    clean_df["image_id"].nunique()
)


print(
    "\n✓ DETECTIONS CLEANED"
)

CLEANING DETECTION RECORDS

Original records: 496803
Valid detection records: 496797
Removed records: 6

Images with valid detections: 39868

✓ DETECTIONS CLEANED


In [ ]:
# ============================================================
# CELL 6 — CONFIDENCE ANALYSIS
# ============================================================

print("=" * 70)
print("DETECTION CONFIDENCE ANALYSIS")
print("=" * 70)


print(
    clean_df["confidence"].describe()
)


thresholds = [
    0.20,
    0.30,
    0.40,
    0.50,
    0.60
]


print(
    "\n" + "-" * 70
)

print(
    "THRESHOLD ANALYSIS"
)

print(
    "-" * 70
)


for threshold in thresholds:

    temp = clean_df[
        clean_df["confidence"] >= threshold
    ]


    detected_images = (
        temp["image_id"]
        .nunique()
    )


    total_records = len(
        temp
    )


    avg_objects = (
        total_records
        / captions_df[
            "image_id"
        ].nunique()
    )


    coverage = (
        detected_images
        /
        captions_df[
            "image_id"
        ].nunique()
        * 100
    )


    print(
        f"\nThreshold: {threshold:.2f}"
    )

    print(
        f"Records: {total_records:,}"
    )

    print(
        f"Images detected: {detected_images:,}"
    )

    print(
        f"Coverage: {coverage:.4f}%"
    )

    print(
        f"Average objects/image: {avg_objects:.4f}"
    )


print(
    "\n✓ CONFIDENCE ANALYSIS COMPLETE"
)

DETECTION CONFIDENCE ANALYSIS
count    496797.000000
mean          0.521480
std           0.245220
min           0.250000
25%           0.306366
50%           0.423132
75%           0.761513
max           0.986814
Name: confidence, dtype: float64

----------------------------------------------------------------------
THRESHOLD ANALYSIS
----------------------------------------------------------------------

Threshold: 0.20
Records: 496,797
Images detected: 39,868
Coverage: 99.9850%
Average objects/image: 12.4592

Threshold: 0.30
Records: 383,229
Images detected: 39,864
Coverage: 99.9749%
Average objects/image: 9.6110

Threshold: 0.40
Records: 265,377
Images detected: 39,842
Coverage: 99.9197%
Average objects/image: 6.6554

Threshold: 0.50
Records: 205,476
Images detected: 39,789
Coverage: 99.7868%
Average objects/image: 5.1531

Threshold: 0.60
Records: 168,124
Images detected: 39,636
Coverage: 99.4031%
Average objects/image: 4.2164

✓ CONFIDENCE ANALYSIS COMPLETE


In [ ]:
# ============================================================
# CELL 7 — STRUCTURED FEATURE CONFIGURATION
# ============================================================

print("=" * 70)
print("STRUCTURED FEATURE CONFIGURATION")
print("=" * 70)


CONFIDENCE_THRESHOLD = 0.30

TOP_K = 10


print(
    "\nConfidence threshold:",
    CONFIDENCE_THRESHOLD
)


print(
    "Maximum objects per image:",
    TOP_K
)


# ------------------------------------------------------------
# APPLY CONFIDENCE FILTER
# ------------------------------------------------------------

filtered_df = clean_df[
    clean_df["confidence"]
    >= CONFIDENCE_THRESHOLD
].copy()


print(
    "\nRecords after confidence filtering:",
    len(filtered_df)
)


print(
    "Images represented:",
    filtered_df["image_id"].nunique()
)


print(
    "\n✓ CONFIGURATION READY"
)

STRUCTURED FEATURE CONFIGURATION

Confidence threshold: 0.3
Maximum objects per image: 10

Records after confidence filtering: 383229
Images represented: 39864

✓ CONFIGURATION READY


In [ ]:
# ============================================================
# CELL 8 — TOP-K OBJECT SELECTION
# ============================================================

print("=" * 70)
print("SELECTING TOP-K OBJECTS PER IMAGE")
print("=" * 70)


# ------------------------------------------------------------
# SORT BY IMAGE + CONFIDENCE
# ------------------------------------------------------------

filtered_df = (
    filtered_df
    .sort_values(
        [
            "image_id",
            "confidence"
        ],
        ascending=[
            True,
            False
        ]
    )
)


# ------------------------------------------------------------
# TOP K
# ------------------------------------------------------------

topk_df = (
    filtered_df
    .groupby(
        "image_id",
        sort=False
    )
    .head(TOP_K)
    .copy()
)


print(
    "\nDetection records after Top-K:",
    len(topk_df)
)


print(
    "Images represented:",
    topk_df["image_id"].nunique()
)


print(
    "\nMaximum objects/image:"
)


print(
    topk_df
    .groupby("image_id")
    .size()
    .max()
)


print(
    "\n✓ TOP-K SELECTION COMPLETE"
)

SELECTING TOP-K OBJECTS PER IMAGE

Detection records after Top-K: 243482
Images represented: 39864

Maximum objects/image:
10

✓ TOP-K SELECTION COMPLETE


In [ ]:
# ============================================================
# CELL 9 — OBJECT VOCABULARY
# ============================================================

print("=" * 70)
print("BUILDING OBJECT VOCABULARY")
print("=" * 70)


object_classes = sorted(
    topk_df[
        "object_name"
    ]
    .unique()
    .tolist()
)


# ------------------------------------------------------------
# SPECIAL TOKENS
# ------------------------------------------------------------

OBJECT_PAD = 0
OBJECT_UNK = 1


object_to_id = {

    "<PAD>":
        OBJECT_PAD,

    "<UNK>":
        OBJECT_UNK
}


next_object_id = 2


for object_name in object_classes:

    object_to_id[
        object_name
    ] = next_object_id

    next_object_id += 1


id_to_object = {
    value: key
    for key, value
    in object_to_id.items()
}


print(
    "\nUnique detected classes:",
    len(object_classes)
)


print(
    "Vocabulary size including special tokens:",
    len(object_to_id)
)


print(
    "\nSample mapping:"
)


for object_name in object_classes[:20]:

    print(
        f"{object_name:20s} → "
        f"{object_to_id[object_name]}"
    )


# ------------------------------------------------------------
# SAVE VOCABULARY
# ------------------------------------------------------------

OBJECT_VOCAB_FILE = (
    OUTPUT_DIR
    / "object_vocabulary.json"
)


with open(
    OBJECT_VOCAB_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "object_to_id":
                object_to_id,

            "id_to_object":
                {
                    str(k): v
                    for k, v
                    in id_to_object.items()
                }
        },
        f,
        indent=4
    )


print(
    "\n✓ OBJECT VOCABULARY SAVED"
)


print(
    OBJECT_VOCAB_FILE
)

BUILDING OBJECT VOCABULARY

Unique detected classes: 80
Vocabulary size including special tokens: 82

Sample mapping:
airplane             → 2
apple                → 3
backpack             → 4
banana               → 5
baseball bat         → 6
baseball glove       → 7
bear                 → 8
bed                  → 9
bench                → 10
bicycle              → 11
bird                 → 12
boat                 → 13
book                 → 14
bottle               → 15
bowl                 → 16
broccoli             → 17
bus                  → 18
cake                 → 19
car                  → 20
carrot               → 21

✓ OBJECT VOCABULARY SAVED
/content/drive/MyDrive/Image_captioning/Dataset/Processed/Structured_Object_Features/object_vocabulary.json


In [ ]:
# ============================================================================
# NOTEBOOK 10 — STRUCTURED OBJECT + CAPTION PREPARATION
# ============================================================================
# RT-DETR-X → confidence filtering → Top-K objects
# → object IDs + spatial features + confidence
# → image-level train/val/test split
# → caption vocabulary + sequences
#
# DATA:
#   Flickr8K + Flickr30K
#
# DETECTOR:
#   RT-DETR-X
#
# IMPORTANT:
#   Images are resolved using IMAGE_NAME from the caption CSV.
#   This avoids image_id formatting mismatches.
# ============================================================================

import os
import re
import json
import zipfile
import shutil
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

print("=" * 75)
print("NOTEBOOK 10 — STRUCTURED OBJECT + CAPTION PREPARATION")
print("=" * 75)


# ============================================================================
# 1. GOOGLE DRIVE
# ============================================================================

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

print("\nGoogle Drive mounted.")


# ============================================================================
# 2. PATHS
# ============================================================================

BASE_DIR = Path("/content/drive/MyDrive/Image_captioning")

ZIP_DIR = BASE_DIR / "Dataset" / "ZIP"
CAPTION_DIR = BASE_DIR / "Dataset" / "Processed" / "Captions"
OUTPUT_DIR = BASE_DIR / "Outputs" / "Structured_Captioning"

RTDETR_FILE = (
    BASE_DIR
    / "Outputs"
    / "RTDETR_Detection"
    / "Final"
    / "rtdetr_x_detections_full.csv"
)

LOCAL_ROOT = Path("/content/Image_captioning_Local")

FLICKR8K_LOCAL = LOCAL_ROOT / "Flickr8K"
FLICKR30K_LOCAL = LOCAL_ROOT / "Flickr30K"

CAPTION_FILE = CAPTION_DIR / "captions_processed.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================================
# 3. EXPERIMENT CONFIGURATION
# ============================================================================

CONF_THRESHOLD = 0.30
TOP_K = 10

MAX_LEN = 32
MIN_WORD_FREQ = 2

TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

SEED = 42

print("\n" + "=" * 75)
print("CONFIGURATION")
print("=" * 75)

print(f"Confidence threshold : {CONF_THRESHOLD}")
print(f"Top-K objects        : {TOP_K}")
print(f"Maximum caption len  : {MAX_LEN}")
print(f"Minimum word freq    : {MIN_WORD_FREQ}")
print(f"Train / Val / Test   : {TRAIN_RATIO} / {VAL_RATIO} / {TEST_RATIO}")
print(f"Random seed          : {SEED}")


# ============================================================================
# 4. FIND ZIP FILES ROBUSTLY
# ============================================================================

print("\n" + "=" * 75)
print("ZIP FILE CHECK")
print("=" * 75)

zip_files = list(ZIP_DIR.glob("*.zip"))

flickr8k_zip = None
flickr30k_zip = None

for z in zip_files:
    name = z.name.lower()

    if "flickr8k" in name:
        flickr8k_zip = z

    elif "flickr30k" in name:
        flickr30k_zip = z

print(f"Flickr8K: {flickr8k_zip}")
print(f"Exists: {flickr8k_zip.exists() if flickr8k_zip else False}")

print()

print(f"Flickr30K: {flickr30k_zip}")
print(f"Exists: {flickr30k_zip.exists() if flickr30k_zip else False}")

assert flickr8k_zip is not None and flickr8k_zip.exists()
assert flickr30k_zip is not None and flickr30k_zip.exists()


# ============================================================================
# 5. EXTRACT DATASETS LOCALLY
# ============================================================================

LOCAL_ROOT.mkdir(parents=True, exist_ok=True)


def count_jpgs(folder):
    if not folder.exists():
        return 0

    count = 0

    for p in folder.rglob("*"):
        if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png"]:
            count += 1

    return count


def extract_if_needed(zip_path, destination, dataset_name):

    destination.mkdir(parents=True, exist_ok=True)

    existing = count_jpgs(destination)

    if existing > 0:
        print(f"\nAlready extracted:")
        print(destination)
        print(f"Image files: {existing:,}")
        return

    print(f"\nExtracting {dataset_name}...")
    print(f"ZIP: {zip_path}")

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(destination)

    existing = count_jpgs(destination)

    print(f"Extracted image files: {existing:,}")


extract_if_needed(
    flickr8k_zip,
    FLICKR8K_LOCAL,
    "Flickr8K"
)

extract_if_needed(
    flickr30k_zip,
    FLICKR30K_LOCAL,
    "Flickr30K"
)


# ============================================================================
# 6. BUILD LOCAL IMAGE INDEX
# ============================================================================

print("\n" + "=" * 75)
print("BUILDING LOCAL IMAGE INDEX")
print("=" * 75)


def collect_images(root):
    files = []

    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in [
            ".jpg",
            ".jpeg",
            ".png"
        ]:
            files.append(p)

    return files


flickr8k_images = collect_images(FLICKR8K_LOCAL)
flickr30k_images = collect_images(FLICKR30K_LOCAL)

all_local_images = flickr8k_images + flickr30k_images

print(f"Flickr8K image files : {len(flickr8k_images):,}")
print(f"Flickr30K image files: {len(flickr30k_images):,}")


# ============================================================================
# 7. CREATE MULTIPLE ROBUST IMAGE LOOKUP KEYS
# ============================================================================

def normalize_filename(value):

    if pd.isna(value):
        return ""

    value = str(value).strip()

    # Convert Windows / Unix paths
    value = value.replace("\\", "/")

    # Keep only filename
    value = value.split("/")[-1]

    # Lowercase
    value = value.lower()

    return value


def normalize_stem(value):

    value = normalize_filename(value)

    if not value:
        return ""

    return Path(value).stem.lower()


# Map:
#   filename.jpg → path
#   filename     → path

image_by_filename = {}
image_by_stem = {}

for path in all_local_images:

    filename = normalize_filename(path.name)
    stem = normalize_stem(path.name)

    image_by_filename[filename] = str(path)
    image_by_stem[stem] = str(path)


print(f"Filename lookup keys: {len(image_by_filename):,}")
print(f"Stem lookup keys    : {len(image_by_stem):,}")

print(f"Unique local images : {len(all_local_images):,}")


# ============================================================================
# 8. LOAD CAPTIONS
# ============================================================================

print("\n" + "=" * 75)
print("LOADING CAPTIONS")
print("=" * 75)

assert CAPTION_FILE.exists(), f"Caption file not found: {CAPTION_FILE}"

captions_raw = pd.read_csv(CAPTION_FILE)

print(f"Original shape: {captions_raw.shape}")

print("Original columns:")
print(list(captions_raw.columns))


# ============================================================================
# 9. EXPLICITLY SELECT REQUIRED COLUMNS
# ============================================================================

required_columns = [
    "image_id",
    "image_name",
    "processed_caption"
]

missing_columns = [
    c for c in required_columns
    if c not in captions_raw.columns
]

assert not missing_columns, (
    f"Missing required caption columns: {missing_columns}"
)


captions_df = captions_raw[
    required_columns
].copy()


captions_df = captions_df.rename(
    columns={
        "processed_caption": "caption"
    }
)


# ============================================================================
# 10. CLEAN CAPTIONS
# ============================================================================

captions_df["image_id"] = (
    captions_df["image_id"]
    .astype(str)
    .str.strip()
)

captions_df["image_name"] = (
    captions_df["image_name"]
    .astype(str)
    .str.strip()
)

captions_df["caption"] = (
    captions_df["caption"]
    .astype(str)
    .str.strip()
)


captions_df = captions_df[
    (captions_df["caption"] != "") &
    (captions_df["caption"].str.lower() != "nan")
].copy()


print("\nClean caption columns:")
print(list(captions_df.columns))

print(f"\nCaption rows: {len(captions_df):,}")
print(
    f"Captioned images: "
    f"{captions_df['image_id'].nunique():,}"
)


# ============================================================================
# 11. RESOLVE IMAGE PATHS
# ============================================================================

print("\n" + "=" * 75)
print("RESOLVING IMAGE PATHS")
print("=" * 75)


def resolve_image(row):

    # ------------------------------------------------------------
    # FIRST PRIORITY: image_name
    # ------------------------------------------------------------

    image_name = normalize_filename(row["image_name"])

    if image_name in image_by_filename:
        return image_by_filename[image_name]

    image_stem = normalize_stem(row["image_name"])

    if image_stem in image_by_stem:
        return image_by_stem[image_stem]


    # ------------------------------------------------------------
    # SECOND PRIORITY: image_id
    # ------------------------------------------------------------

    image_id = normalize_filename(row["image_id"])

    if image_id in image_by_filename:
        return image_by_filename[image_id]

    image_id_stem = normalize_stem(row["image_id"])

    if image_id_stem in image_by_stem:
        return image_by_stem[image_id_stem]


    # ------------------------------------------------------------
    # NOTHING FOUND
    # ------------------------------------------------------------

    return None


tqdm.pandas(desc="Resolving image paths")

captions_df["image_path"] = captions_df.progress_apply(
    resolve_image,
    axis=1
)


missing = captions_df["image_path"].isna().sum()

print(f"\nMissing image paths: {missing:,}")


# ============================================================================
# 12. DEBUG IF ANY IMAGES ARE MISSING
# ============================================================================

if missing > 0:

    print("\nWARNING: Some images could not be resolved.")

    print("\nExamples from CSV:")

    print(
        captions_df[
            captions_df["image_path"].isna()
        ][
            ["image_id", "image_name"]
        ]
        .drop_duplicates()
        .head(20)
        .to_string(index=False)
    )

    print("\nExamples of local image files:")

    for p in all_local_images[:20]:
        print(p.name)


assert missing == 0, (
    f"{missing:,} caption rows could not be mapped "
    f"to local images."
)


# ============================================================================
# 13. VERIFY IMAGE COVERAGE
# ============================================================================

caption_image_ids = set(
    captions_df["image_id"].unique()
)

resolved_image_names = set(
    captions_df["image_path"].map(
        lambda x: Path(x).name
    ).unique()
)

print("\nImage resolution successful.")

print(
    f"Unique caption images: "
    f"{len(caption_image_ids):,}"
)

print(
    f"Unique resolved image files: "
    f"{len(resolved_image_names):,}"
)


# ============================================================================
# 14. LOAD RT-DETR-X DETECTIONS
# ============================================================================

print("\n" + "=" * 75)
print("LOADING RT-DETR-X DETECTIONS")
print("=" * 75)

assert RTDETR_FILE.exists(), (
    f"RT-DETR-X file not found:\n{RTDETR_FILE}"
)

detections = pd.read_csv(RTDETR_FILE)

print(f"Detection shape: {detections.shape}")

print("Detection columns:")
print(list(detections.columns))


# ============================================================================
# 15. NORMALIZE DETECTION COLUMN NAMES
# ============================================================================

detections.columns = [
    str(c).strip().lower().replace(" ", "_")
    for c in detections.columns
]


def find_column(possible_names):

    for name in possible_names:
        if name in detections.columns:
            return name

    return None


IMAGE_COL = find_column([
    "image_id",
    "image_name",
    "filename",
    "file_name"
])

OBJECT_COL = find_column([
    "class_name",
    "object_name",
    "name",
    "class"
])

CONF_COL = find_column([
    "confidence",
    "conf",
    "score"
])

X1_COL = find_column([
    "xmin",
    "x1",
    "box_x1"
])

Y1_COL = find_column([
    "ymin",
    "y1",
    "box_y1"
])

X2_COL = find_column([
    "xmax",
    "x2",
    "box_x2"
])

Y2_COL = find_column([
    "ymax",
    "y2",
    "box_y2"
])


print("\nDetected columns:")

print("IMAGE :", IMAGE_COL)
print("OBJECT:", OBJECT_COL)
print("CONF  :", CONF_COL)
print("X1    :", X1_COL)
print("Y1    :", Y1_COL)
print("X2    :", X2_COL)
print("Y2    :", Y2_COL)


assert IMAGE_COL is not None
assert OBJECT_COL is not None
assert CONF_COL is not None
assert X1_COL is not None
assert Y1_COL is not None
assert X2_COL is not None
assert Y2_COL is not None


# ============================================================================
# 16. STANDARDIZE DETECTION DATA
# ============================================================================

det = detections.rename(
    columns={
        IMAGE_COL: "image_id",
        OBJECT_COL: "object_name",
        CONF_COL: "confidence",
        X1_COL: "xmin",
        Y1_COL: "ymin",
        X2_COL: "xmax",
        Y2_COL: "ymax"
    }
)[
    [
        "image_id",
        "object_name",
        "confidence",
        "xmin",
        "ymin",
        "xmax",
        "ymax"
    ]
].copy()


det["image_id"] = (
    det["image_id"]
    .astype(str)
    .str.strip()
)

det["object_name"] = (
    det["object_name"]
    .astype(str)
    .str.strip()
)

for col in [
    "confidence",
    "xmin",
    "ymin",
    "xmax",
    "ymax"
]:
    det[col] = pd.to_numeric(
        det[col],
        errors="coerce"
    )


# ============================================================================
# 17. FILTER VALID DETECTIONS
# ============================================================================

det = det[
    det["object_name"].notna() &
    (det["object_name"] != "") &
    (det["object_name"].str.lower() != "nan")
].copy()

det = det[
    det["confidence"] >= CONF_THRESHOLD
].copy()

det = det.dropna(
    subset=[
        "xmin",
        "ymin",
        "xmax",
        "ymax",
        "confidence"
    ]
).copy()

print(
    f"\nValid detections after "
    f"confidence filtering: {len(det):,}"
)


# ============================================================================
# 18. NORMALIZE DETECTION IMAGE IDs
# ============================================================================
# RT-DETR CSV may store image names differently.
# Create a normalized filename/stem to match captions robustly.

det["image_filename"] = det["image_id"].map(
    normalize_filename
)

det["image_stem"] = det["image_id"].map(
    normalize_stem
)


# ============================================================================
# 19. MATCH DETECTIONS TO CAPTION IMAGE NAMES
# ============================================================================

caption_lookup = captions_df[
    [
        "image_id",
        "image_name",
        "image_path"
    ]
].drop_duplicates().copy()


caption_lookup["image_filename"] = (
    caption_lookup["image_name"]
    .map(normalize_filename)
)

caption_lookup["image_stem"] = (
    caption_lookup["image_name"]
    .map(normalize_stem)
)


# filename → canonical image_id
filename_to_image_id = dict(
    zip(
        caption_lookup["image_filename"],
        caption_lookup["image_id"]
    )
)

stem_to_image_id = dict(
    zip(
        caption_lookup["image_stem"],
        caption_lookup["image_id"]
    )
)


def resolve_detection_image_id(value):

    filename = normalize_filename(value)

    if filename in filename_to_image_id:
        return filename_to_image_id[filename]

    stem = normalize_stem(value)

    if stem in stem_to_image_id:
        return stem_to_image_id[stem]

    return None


tqdm.pandas(desc="Matching detections")

det["canonical_image_id"] = det["image_id"].progress_map(
    resolve_detection_image_id
)


matched_det = det[
    det["canonical_image_id"].notna()
].copy()


print(
    f"\nMatched detections: "
    f"{len(matched_det):,}"
)

print(
    f"Unmatched detections: "
    f"{len(det) - len(matched_det):,}"
)


# ============================================================================
# 20. NORMALIZE BOUNDING BOXES
# ============================================================================

# Create image dimensions lookup

print("\nReading image dimensions...")

dimension_cache = {}

unique_paths = captions_df[
    ["image_id", "image_path"]
].drop_duplicates()


for row in tqdm(
    unique_paths.itertuples(index=False),
    total=len(unique_paths),
    desc="Image dimensions"
):

    image_id = row.image_id
    path = row.image_path

    try:

        from PIL import Image

        with Image.open(path) as img:
            width, height = img.size

        dimension_cache[image_id] = (
            width,
            height
        )

    except Exception:

        dimension_cache[image_id] = (
            1,
            1
        )


# Map dimensions

matched_det["width"] = matched_det[
    "canonical_image_id"
].map(
    lambda x: dimension_cache.get(x, (1, 1))[0]
)

matched_det["height"] = matched_det[
    "canonical_image_id"
].map(
    lambda x: dimension_cache.get(x, (1, 1))[1]
)


# Avoid division by zero

matched_det["width"] = matched_det[
    "width"
].clip(lower=1)

matched_det["height"] = matched_det[
    "height"
].clip(lower=1)


# ============================================================================
# 21. CLIP BOX COORDINATES TO IMAGE
# ============================================================================

matched_det["xmin"] = matched_det["xmin"].clip(
    lower=0
)

matched_det["ymin"] = matched_det["ymin"].clip(
    lower=0
)

matched_det["xmax"] = matched_det["xmax"].clip(
    lower=0
)

matched_det["ymax"] = matched_det["ymax"].clip(
    lower=0
)

matched_det["xmin"] = np.minimum(
    matched_det["xmin"],
    matched_det["width"]
)

matched_det["xmax"] = np.minimum(
    matched_det["xmax"],
    matched_det["width"]
)

matched_det["ymin"] = np.minimum(
    matched_det["ymin"],
    matched_det["height"]
)

matched_det["ymax"] = np.minimum(
    matched_det["ymax"],
    matched_det["height"]
)


# ============================================================================
# 22. CREATE NORMALIZED SPATIAL FEATURES
# ============================================================================

matched_det["x1_norm"] = (
    matched_det["xmin"] /
    matched_det["width"]
)

matched_det["y1_norm"] = (
    matched_det["ymin"] /
    matched_det["height"]
)

matched_det["x2_norm"] = (
    matched_det["xmax"] /
    matched_det["width"]
)

matched_det["y2_norm"] = (
    matched_det["ymax"] /
    matched_det["height"]
)


matched_det["cx_norm"] = (
    matched_det["x1_norm"] +
    matched_det["x2_norm"]
) / 2


matched_det["cy_norm"] = (
    matched_det["y1_norm"] +
    matched_det["y2_norm"]
) / 2


matched_det["box_width"] = (
    matched_det["x2_norm"] -
    matched_det["x1_norm"]
).clip(lower=0)


matched_det["box_height"] = (
    matched_det["y2_norm"] -
    matched_det["y1_norm"]
).clip(lower=0)


matched_det["box_area"] = (
    matched_det["box_width"] *
    matched_det["box_height"]
)


# ============================================================================
# 23. TOP-K OBJECTS PER IMAGE
# ============================================================================

print("\n" + "=" * 75)
print("SELECTING TOP-K OBJECTS")
print("=" * 75)

matched_det = matched_det.sort_values(
    [
        "canonical_image_id",
        "confidence"
    ],
    ascending=[
        True,
        False
    ]
)

top_det = (
    matched_det
    .groupby(
        "canonical_image_id",
        sort=False
    )
    .head(TOP_K)
    .copy()
)

print(
    f"Detections retained: "
    f"{len(top_det):,}"
)

print(
    f"Images with ≥1 object: "
    f"{top_det['canonical_image_id'].nunique():,}"
)


# ============================================================================
# 24. BUILD OBJECT VOCABULARY
# ============================================================================

print("\n" + "=" * 75)
print("BUILDING OBJECT VOCABULARY")
print("=" * 75)

object_counter = Counter(
    top_det["object_name"].str.lower()
)

object_vocabulary = {
    "<pad>": 0,
    "<unk>": 1
}

for obj, _ in sorted(
    object_counter.items(),
    key=lambda x: (-x[1], x[0])
):

    if obj not in object_vocabulary:
        object_vocabulary[obj] = (
            len(object_vocabulary)
        )


print(
    f"Object vocabulary size: "
    f"{len(object_vocabulary):,}"
)


# ============================================================================
# 25. PREPARE IMAGE-LEVEL OBJECT ARRAYS
# ============================================================================

print("\n" + "=" * 75)
print("BUILDING STRUCTURED OBJECT ARRAYS")
print("=" * 75)

all_image_ids = (
    captions_df["image_id"]
    .drop_duplicates()
    .tolist()
)

N_IMAGES = len(all_image_ids)

print(f"Total images: {N_IMAGES:,}")


image_to_index = {
    image_id: idx
    for idx, image_id in enumerate(all_image_ids)
}


# Object IDs
object_ids = np.zeros(
    (N_IMAGES, TOP_K),
    dtype=np.int32
)

# Spatial features:
# xmin, ymin, xmax, ymax,
# center_x, center_y,
# width, height, area

spatial_features = np.zeros(
    (N_IMAGES, TOP_K, 9),
    dtype=np.float32
)

# Confidence
confidence_features = np.zeros(
    (N_IMAGES, TOP_K),
    dtype=np.float32
)

# Mask
object_mask = np.zeros(
    (N_IMAGES, TOP_K),
    dtype=np.bool_
)


grouped = top_det.groupby(
    "canonical_image_id"
)


for image_id, group in tqdm(
    grouped,
    total=grouped.ngroups,
    desc="Creating object tensors"
):

    if image_id not in image_to_index:
        continue

    idx = image_to_index[image_id]

    group = group.reset_index(drop=True)

    for k, row in group.iterrows():

        if k >= TOP_K:
            break

        object_name = (
            str(row["object_name"])
            .strip()
            .lower()
        )

        object_id = object_vocabulary.get(
            object_name,
            object_vocabulary["<unk>"]
        )

        object_ids[idx, k] = object_id

        spatial_features[idx, k] = [
            row["x1_norm"],
            row["y1_norm"],
            row["x2_norm"],
            row["y2_norm"],
            row["cx_norm"],
            row["cy_norm"],
            row["box_width"],
            row["box_height"],
            row["box_area"]
        ]

        confidence_features[idx, k] = (
            float(row["confidence"])
        )

        object_mask[idx, k] = True


print("\nArray shapes:")

print(
    "object_ids:",
    object_ids.shape
)

print(
    "spatial_features:",
    spatial_features.shape
)

print(
    "confidence_features:",
    confidence_features.shape
)

print(
    "object_mask:",
    object_mask.shape
)


# ============================================================================
# 26. IMAGE-LEVEL TRAIN / VAL / TEST SPLIT
# ============================================================================

print("\n" + "=" * 75)
print("IMAGE-LEVEL DATASET SPLIT")
print("=" * 75)

rng = np.random.default_rng(SEED)

shuffled_ids = np.array(
    all_image_ids,
    dtype=object
)

rng.shuffle(shuffled_ids)

n_train = int(
    N_IMAGES * TRAIN_RATIO
)

n_val = int(
    N_IMAGES * VAL_RATIO
)

train_ids = shuffled_ids[
    :n_train
]

val_ids = shuffled_ids[
    n_train:n_train + n_val
]

test_ids = shuffled_ids[
    n_train + n_val:
]


print(f"Train images: {len(train_ids):,}")
print(f"Val images  : {len(val_ids):,}")
print(f"Test images : {len(test_ids):,}")

assert (
    len(train_ids) +
    len(val_ids) +
    len(test_ids)
) == N_IMAGES

assert len(
    set(train_ids) &
    set(val_ids)
) == 0

assert len(
    set(train_ids) &
    set(test_ids)
) == 0

assert len(
    set(val_ids) &
    set(test_ids)
) == 0


# ============================================================================
# 27. CAPTION TOKENIZATION
# ============================================================================

print("\n" + "=" * 75)
print("BUILDING CAPTION VOCABULARY")
print("=" * 75)


def tokenize_caption(text):

    text = str(text).lower().strip()

    # Keep words/numbers/basic apostrophes
    tokens = re.findall(
        r"[a-z0-9]+(?:'[a-z0-9]+)?",
        text
    )

    return tokens


captions_df["tokens"] = (
    captions_df["caption"]
    .map(tokenize_caption)
)


# ============================================================================
# 28. DETERMINE TRAINING CAPTIONS
# ============================================================================

train_id_set = set(train_ids)

train_captions = captions_df[
    captions_df["image_id"].isin(
        train_id_set
    )
].copy()


print(
    f"Training caption rows: "
    f"{len(train_captions):,}"
)


# ============================================================================
# 29. CAPTION WORD FREQUENCIES
# ============================================================================

word_counter = Counter()

for tokens in train_captions["tokens"]:
    word_counter.update(tokens)


caption_vocabulary = {
    "<pad>": 0,
    "<start>": 1,
    "<end>": 2,
    "<unk>": 3
}


for word, count in sorted(
    word_counter.items(),
    key=lambda x: (-x[1], x[0])
):

    if count >= MIN_WORD_FREQ:

        if word not in caption_vocabulary:

            caption_vocabulary[word] = (
                len(caption_vocabulary)
            )


print(
    f"Caption vocabulary size: "
    f"{len(caption_vocabulary):,}"
)


# ============================================================================
# 30. ENCODE CAPTIONS
# ============================================================================

print("\n" + "=" * 75)
print("ENCODING CAPTION SEQUENCES")
print("=" * 75)


PAD_ID = caption_vocabulary["<pad>"]
START_ID = caption_vocabulary["<start>"]
END_ID = caption_vocabulary["<end>"]
UNK_ID = caption_vocabulary["<unk>"]


def encode_caption(tokens):

    sequence = [
        START_ID
    ]

    for token in tokens:

        sequence.append(
            caption_vocabulary.get(
                token,
                UNK_ID
            )
        )

    sequence.append(
        END_ID
    )

    # Truncate
    sequence = sequence[
        :MAX_LEN
    ]

    # Pad
    if len(sequence) < MAX_LEN:

        sequence += [
            PAD_ID
        ] * (
            MAX_LEN - len(sequence)
        )

    return sequence


caption_sequences = np.array(
    [
        encode_caption(tokens)
        for tokens in captions_df["tokens"]
    ],
    dtype=np.int32
)


print(
    f"Caption sequence shape: "
    f"{caption_sequences.shape}"
)


# ============================================================================
# 31. SAVE IMAGE METADATA
# ============================================================================

print("\n" + "=" * 75)
print("SAVING OUTPUTS")
print("=" * 75)


metadata_df = captions_df[
    [
        "image_id",
        "image_name",
        "image_path"
    ]
].drop_duplicates(
    subset=["image_id"]
).copy()


metadata_df["object_count"] = (
    object_mask.sum(axis=1)
)


metadata_df["has_objects"] = (
    metadata_df["object_count"] > 0
)


metadata_df.to_csv(
    OUTPUT_DIR /
    "image_object_metadata.csv",
    index=False
)


# ============================================================================
# 32. SAVE SPLITS
# ============================================================================

pd.DataFrame({
    "image_id": train_ids
}).to_csv(
    OUTPUT_DIR /
    "train_image_ids.csv",
    index=False
)


pd.DataFrame({
    "image_id": val_ids
}).to_csv(
    OUTPUT_DIR /
    "val_image_ids.csv",
    index=False
)


pd.DataFrame({
    "image_id": test_ids
}).to_csv(
    OUTPUT_DIR /
    "test_image_ids.csv",
    index=False
)


# ============================================================================
# 33. SAVE OBJECT VOCABULARY
# ============================================================================

with open(
    OUTPUT_DIR /
    "object_vocabulary.json",
    "w"
) as f:

    json.dump(
        object_vocabulary,
        f,
        indent=2
    )


# ============================================================================
# 34. SAVE CAPTION VOCABULARY
# ============================================================================

with open(
    OUTPUT_DIR /
    "caption_vocabulary.json",
    "w"
) as f:

    json.dump(
        caption_vocabulary,
        f,
        indent=2
    )


# ============================================================================
# 35. SAVE STRUCTURED OBJECT FEATURES
# ============================================================================

np.savez_compressed(
    OUTPUT_DIR /
    "rtdetr_structured_object_features.npz",

    image_ids=np.array(
        all_image_ids,
        dtype=object
    ),

    object_ids=object_ids,

    spatial_features=spatial_features,

    confidence_features=confidence_features,

    object_mask=object_mask
)


# ============================================================================
# 36. SAVE CAPTION SEQUENCES
# ============================================================================

np.savez_compressed(
    OUTPUT_DIR /
    "caption_sequences.npz",

    image_ids=np.array(
        captions_df["image_id"],
        dtype=object
    ),

    sequences=caption_sequences
)


# ============================================================================
# 37. SAVE CONFIGURATION
# ============================================================================

config = {

    "dataset": [
        "Flickr8K",
        "Flickr30K"
    ],

    "total_images": int(N_IMAGES),

    "total_captions": int(
        len(captions_df)
    ),

    "detector": "RT-DETR-X",

    "confidence_threshold": CONF_THRESHOLD,

    "top_k": TOP_K,

    "max_caption_length": MAX_LEN,

    "minimum_word_frequency": MIN_WORD_FREQ,

    "train_ratio": TRAIN_RATIO,

    "validation_ratio": VAL_RATIO,

    "test_ratio": TEST_RATIO,

    "random_seed": SEED,

    "spatial_features": [
        "xmin",
        "ymin",
        "xmax",
        "ymax",
        "center_x",
        "center_y",
        "box_width",
        "box_height",
        "box_area"
    ],

    "object_feature_dimensions": {
        "object_id": 1,
        "spatial": 9,
        "confidence": 1
    },

    "special_tokens": {
        "caption_pad": PAD_ID,
        "caption_start": START_ID,
        "caption_end": END_ID,
        "caption_unk": UNK_ID,

        "object_pad": object_vocabulary["<pad>"],
        "object_unk": object_vocabulary["<unk>"]
    }
}


with open(
    OUTPUT_DIR /
    "structured_captioning_config.json",
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=2
    )


# ============================================================================
# 38. FINAL VALIDATION
# ============================================================================

print("\n" + "=" * 75)
print("FINAL VALIDATION")
print("=" * 75)

assert N_IMAGES == 39_874, (
    f"Expected 39,874 images, got {N_IMAGES}"
)

assert len(captions_df) == 199_369, (
    f"Expected 199,369 captions, "
    f"got {len(captions_df)}"
)

assert object_ids.shape == (
    39_874,
    TOP_K
)

assert spatial_features.shape == (
    39_874,
    TOP_K,
    9
)

assert confidence_features.shape == (
    39_874,
    TOP_K
)

assert object_mask.shape == (
    39_874,
    TOP_K
)

assert caption_sequences.shape == (
    199_369,
    MAX_LEN
)

assert (
    len(train_ids) +
    len(val_ids) +
    len(test_ids)
) == 39_874


# ============================================================================
# 39. FINAL SUMMARY
# ============================================================================

print("\n" + "=" * 75)
print("NOTEBOOK 10 COMPLETE")
print("=" * 75)

print("\nDATASET")
print("-" * 75)

print(f"Images                 : {N_IMAGES:,}")
print(f"Captions               : {len(captions_df):,}")

print("\nOBJECT FEATURES")
print("-" * 75)

print(
    f"Object IDs             : "
    f"{object_ids.shape}"
)

print(
    f"Spatial features       : "
    f"{spatial_features.shape}"
)

print(
    f"Confidence features    : "
    f"{confidence_features.shape}"
)

print(
    f"Object mask            : "
    f"{object_mask.shape}"
)

print("\nVOCABULARIES")
print("-" * 75)

print(
    f"Object vocabulary      : "
    f"{len(object_vocabulary):,}"
)

print(
    f"Caption vocabulary     : "
    f"{len(caption_vocabulary):,}"
)

print("\nSPLIT")
print("-" * 75)

print(f"Train                  : {len(train_ids):,}")
print(f"Validation             : {len(val_ids):,}")
print(f"Test                   : {len(test_ids):,}")

print("\nOUTPUT DIRECTORY")
print("-" * 75)

print(OUTPUT_DIR)

print("\nFiles created:")

for p in sorted(OUTPUT_DIR.iterdir()):
    print(
        f"  {p.name:45s} "
        f"{p.stat().st_size / (1024**2):8.2f} MB"
    )

print("\n" + "=" * 75)
print("READY FOR NOTEBOOK 11 — OBJECT-AWARE TRANSFORMER")
print("=" * 75)

NOTEBOOK 10 — STRUCTURED OBJECT + CAPTION PREPARATION
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Google Drive mounted.

CONFIGURATION
Confidence threshold : 0.3
Top-K objects        : 10
Maximum caption len  : 32
Minimum word freq    : 2
Train / Val / Test   : 0.8 / 0.1 / 0.1
Random seed          : 42

ZIP FILE CHECK
Flickr8K: /content/drive/MyDrive/Image_captioning/Dataset/ZIP/flickr8k.zip
Exists: True

Flickr30K: /content/drive/MyDrive/Image_captioning/Dataset/ZIP/flickr30k.zip
Exists: True

Already extracted:
/content/Image_captioning_Local/Flickr8K
Image files: 8,091

Already extracted:
/content/Image_captioning_Local/Flickr30K
Image files: 63,566

BUILDING LOCAL IMAGE INDEX
Flickr8K image files : 8,091
Flickr30K image files: 63,566
Filename lookup keys: 39,874
Stem lookup keys    : 39,874
Unique local images : 71,657

LOADING CAPTIONS
Original shape: (199369, 7)
Original columns:
['image_id', 'd

Resolving image paths:   0%|          | 0/199369 [00:00<?, ?it/s]


Missing image paths: 0

Image resolution successful.
Unique caption images: 39,874
Unique resolved image files: 39,874

LOADING RT-DETR-X DETECTIONS
Detection shape: (496803, 9)
Detection columns:
['image_id', 'image_path', 'object_name', 'confidence', 'xmin', 'ymin', 'xmax', 'ymax', 'objects_detected']

Detected columns:
IMAGE : image_id
OBJECT: object_name
CONF  : confidence
X1    : xmin
Y1    : ymin
X2    : xmax
Y2    : ymax

Valid detections after confidence filtering: 383,229


Matching detections:   0%|          | 0/383229 [00:00<?, ?it/s]


Matched detections: 383,229
Unmatched detections: 0

Reading image dimensions...


Image dimensions:   0%|          | 0/39874 [00:00<?, ?it/s]


SELECTING TOP-K OBJECTS
Detections retained: 243,482
Images with ≥1 object: 39,864

BUILDING OBJECT VOCABULARY
Object vocabulary size: 82

BUILDING STRUCTURED OBJECT ARRAYS
Total images: 39,874


Creating object tensors:   0%|          | 0/39864 [00:00<?, ?it/s]


Array shapes:
object_ids: (39874, 10)
spatial_features: (39874, 10, 9)
confidence_features: (39874, 10)
object_mask: (39874, 10)

IMAGE-LEVEL DATASET SPLIT
Train images: 31,899
Val images  : 3,987
Test images : 3,988

BUILDING CAPTION VOCABULARY
Training caption rows: 159,494
Caption vocabulary size: 12,162

ENCODING CAPTION SEQUENCES
Caption sequence shape: (199369, 32)

SAVING OUTPUTS

FINAL VALIDATION

NOTEBOOK 10 COMPLETE

DATASET
---------------------------------------------------------------------------
Images                 : 39,874
Captions               : 199,369

OBJECT FEATURES
---------------------------------------------------------------------------
Object IDs             : (39874, 10)
Spatial features       : (39874, 10, 9)
Confidence features    : (39874, 10)
Object mask            : (39874, 10)

VOCABULARIES
---------------------------------------------------------------------------
Object vocabulary      : 82
Caption vocabulary     : 12,162

SPLIT
------------------

In [ ]:
# ============================================================================
# BUILDING STRUCTURED OBJECT REPRESENTATIONS
# ============================================================================

print("=" * 70)
print("BUILDING STRUCTURED OBJECT REPRESENTATIONS")
print("=" * 70)

# --------------------------------------------------------------------------
# 1. MAKE A COPY
# --------------------------------------------------------------------------

topk_df = top_det.copy()


# --------------------------------------------------------------------------
# 2. RESOLVE IMAGE DIMENSION COLUMN NAMES
# --------------------------------------------------------------------------
# Different preparation versions may use:
#   width / height
# or:
#   image_width / image_height
#
# Normalize them here so the rest of the pipeline is independent
# of the column naming convention.
# --------------------------------------------------------------------------

if "image_width" in topk_df.columns:
    WIDTH_COL = "image_width"
elif "width" in topk_df.columns:
    WIDTH_COL = "width"
else:
    raise KeyError(
        "Could not find image width column. "
        f"Available columns: {list(topk_df.columns)}"
    )


if "image_height" in topk_df.columns:
    HEIGHT_COL = "image_height"
elif "height" in topk_df.columns:
    HEIGHT_COL = "height"
else:
    raise KeyError(
        "Could not find image height column. "
        f"Available columns: {list(topk_df.columns)}"
    )


print(f"Using width column : {WIDTH_COL}")
print(f"Using height column: {HEIGHT_COL}")


# --------------------------------------------------------------------------
# 3. ENSURE NUMERIC VALUES
# --------------------------------------------------------------------------

topk_df[WIDTH_COL] = pd.to_numeric(
    topk_df[WIDTH_COL],
    errors="coerce"
)

topk_df[HEIGHT_COL] = pd.to_numeric(
    topk_df[HEIGHT_COL],
    errors="coerce"
)

topk_df["xmin"] = pd.to_numeric(
    topk_df["xmin"],
    errors="coerce"
)

topk_df["ymin"] = pd.to_numeric(
    topk_df["ymin"],
    errors="coerce"
)

topk_df["xmax"] = pd.to_numeric(
    topk_df["xmax"],
    errors="coerce"
)

topk_df["ymax"] = pd.to_numeric(
    topk_df["ymax"],
    errors="coerce"
)

topk_df["confidence"] = pd.to_numeric(
    topk_df["confidence"],
    errors="coerce"
)


# --------------------------------------------------------------------------
# 4. REMOVE INVALID DIMENSIONS / BOXES
# --------------------------------------------------------------------------

topk_df = topk_df.dropna(
    subset=[
        WIDTH_COL,
        HEIGHT_COL,
        "xmin",
        "ymin",
        "xmax",
        "ymax",
        "confidence"
    ]
).copy()

topk_df = topk_df[
    (topk_df[WIDTH_COL] > 0) &
    (topk_df[HEIGHT_COL] > 0)
].copy()


# --------------------------------------------------------------------------
# 5. CLIP BOXES TO IMAGE BOUNDARIES
# --------------------------------------------------------------------------

topk_df["xmin"] = topk_df["xmin"].clip(
    lower=0
)

topk_df["ymin"] = topk_df["ymin"].clip(
    lower=0
)

topk_df["xmax"] = topk_df["xmax"].clip(
    lower=0
)

topk_df["ymax"] = topk_df["ymax"].clip(
    lower=0
)


topk_df["xmin"] = np.minimum(
    topk_df["xmin"],
    topk_df[WIDTH_COL]
)

topk_df["xmax"] = np.minimum(
    topk_df["xmax"],
    topk_df[WIDTH_COL]
)

topk_df["ymin"] = np.minimum(
    topk_df["ymin"],
    topk_df[HEIGHT_COL]
)

topk_df["ymax"] = np.minimum(
    topk_df["ymax"],
    topk_df[HEIGHT_COL]
)


# --------------------------------------------------------------------------
# 6. NORMALIZED BOUNDING BOX
# --------------------------------------------------------------------------

topk_df["nxmin"] = (
    topk_df["xmin"] /
    topk_df[WIDTH_COL]
)

topk_df["nymin"] = (
    topk_df["ymin"] /
    topk_df[HEIGHT_COL]
)

topk_df["nxmax"] = (
    topk_df["xmax"] /
    topk_df[WIDTH_COL]
)

topk_df["nymax"] = (
    topk_df["ymax"] /
    topk_df[HEIGHT_COL]
)


# --------------------------------------------------------------------------
# 7. CENTER COORDINATES
# --------------------------------------------------------------------------

topk_df["center_x"] = (
    topk_df["nxmin"] +
    topk_df["nxmax"]
) / 2.0

topk_df["center_y"] = (
    topk_df["nymin"] +
    topk_df["nymax"]
) / 2.0


# --------------------------------------------------------------------------
# 8. BOX WIDTH / HEIGHT
# --------------------------------------------------------------------------

topk_df["box_width"] = (
    topk_df["nxmax"] -
    topk_df["nxmin"]
).clip(lower=0)


topk_df["box_height"] = (
    topk_df["nymax"] -
    topk_df["nymin"]
).clip(lower=0)


# --------------------------------------------------------------------------
# 9. BOX AREA
# --------------------------------------------------------------------------

topk_df["box_area"] = (
    topk_df["box_width"] *
    topk_df["box_height"]
)


# --------------------------------------------------------------------------
# 10. CREATE FINAL 9-DIMENSIONAL SPATIAL VECTOR
# --------------------------------------------------------------------------
#
# [xmin, ymin, xmax, ymax,
#  center_x, center_y,
#  width, height, area]
#
# All values are normalized to approximately [0, 1].
# --------------------------------------------------------------------------

SPATIAL_COLUMNS = [
    "nxmin",
    "nymin",
    "nxmax",
    "nymax",
    "center_x",
    "center_y",
    "box_width",
    "box_height",
    "box_area"
]


topk_df["spatial_vector"] = topk_df[
    SPATIAL_COLUMNS
].values.tolist()


# --------------------------------------------------------------------------
# 11. CONFIDENCE VECTOR
# --------------------------------------------------------------------------

topk_df["confidence_vector"] = (
    topk_df["confidence"]
    .astype(np.float32)
)


# --------------------------------------------------------------------------
# 12. OBJECT NAME NORMALIZATION
# --------------------------------------------------------------------------

topk_df["object_name_clean"] = (
    topk_df["object_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)


# --------------------------------------------------------------------------
# 13. SORT OBJECTS WITHIN EACH IMAGE
# --------------------------------------------------------------------------
# Highest-confidence object first.
# This gives deterministic Top-K ordering.
# --------------------------------------------------------------------------

topk_df = topk_df.sort_values(
    [
        "canonical_image_id",
        "confidence"
    ],
    ascending=[
        True,
        False
    ]
).copy()


# --------------------------------------------------------------------------
# 14. FINAL CHECKS
# --------------------------------------------------------------------------

print("\nStructured representation created.")

print(
    f"Detection rows: {len(topk_df):,}"
)

print(
    f"Images represented: "
    f"{topk_df['canonical_image_id'].nunique():,}"
)

print(
    f"Spatial feature dimensions: "
    f"{len(SPATIAL_COLUMNS)}"
)

print(
    "\nSpatial columns:"
)

for i, col in enumerate(
    SPATIAL_COLUMNS,
    start=1
):
    print(f"  {i}. {col}")


# --------------------------------------------------------------------------
# 15. VALUE RANGE CHECK
# --------------------------------------------------------------------------

spatial_matrix = topk_df[
    SPATIAL_COLUMNS
].to_numpy(
    dtype=np.float32
)

print("\nSpatial feature range:")

print(
    f"Minimum: {spatial_matrix.min():.6f}"
)

print(
    f"Maximum: {spatial_matrix.max():.6f}"
)


assert np.isfinite(
    spatial_matrix
).all()

assert spatial_matrix.min() >= 0.0
assert spatial_matrix.max() <= 1.0 + 1e-6


print("\n" + "=" * 70)
print("STRUCTURED OBJECT REPRESENTATIONS READY")
print("=" * 70)

BUILDING STRUCTURED OBJECT REPRESENTATIONS
Using width column : width
Using height column: height

Structured representation created.
Detection rows: 243,482
Images represented: 39,864
Spatial feature dimensions: 9

Spatial columns:
  1. nxmin
  2. nymin
  3. nxmax
  4. nymax
  5. center_x
  6. center_y
  7. box_width
  8. box_height
  9. box_area

Spatial feature range:
Minimum: 0.000000
Maximum: 1.000000

STRUCTURED OBJECT REPRESENTATIONS READY


In [ ]:
# ============================================================================
# CELL 12 — BUILD FIXED-SIZE OBJECT TENSORS
# ============================================================================

print("=" * 70)
print("BUILDING FIXED-SIZE OBJECT TENSORS")
print("=" * 70)

# --------------------------------------------------------------------------
# REQUIREMENTS FROM PREVIOUS CELLS
# --------------------------------------------------------------------------

assert "topk_df" in globals(), (
    "topk_df not found. Run the structured object representation cell first."
)

assert "all_image_ids" in globals(), (
    "all_image_ids not found."
)

assert "object_vocabulary" in globals(), (
    "object_vocabulary not found."
)

assert "TOP_K" in globals(), (
    "TOP_K not found."
)

assert "SPATIAL_COLUMNS" in globals(), (
    "SPATIAL_COLUMNS not found."
)


# --------------------------------------------------------------------------
# IMAGE SETUP
# --------------------------------------------------------------------------

all_image_ids = list(all_image_ids)

N_IMAGES = len(all_image_ids)

image_to_index = {
    image_id: idx
    for idx, image_id in enumerate(all_image_ids)
}


print(f"\nTotal images: {N_IMAGES}")
print(f"TOP_K: {TOP_K}")


# --------------------------------------------------------------------------
# CREATE FIXED-SIZE ARRAYS
# --------------------------------------------------------------------------
#
# object_ids:
#     [N_IMAGES, TOP_K]
#
# spatial_features:
#     [N_IMAGES, TOP_K, 9]
#
# confidence_features:
#     [N_IMAGES, TOP_K]
#
# object_mask:
#     [N_IMAGES, TOP_K]
#
# All arrays are initialized to zero, so images without detections
# automatically receive padded object slots.
# --------------------------------------------------------------------------

object_ids = np.zeros(
    (N_IMAGES, TOP_K),
    dtype=np.int32
)

spatial_features = np.zeros(
    (N_IMAGES, TOP_K, 9),
    dtype=np.float32
)

confidence_features = np.zeros(
    (N_IMAGES, TOP_K),
    dtype=np.float32
)

object_mask = np.zeros(
    (N_IMAGES, TOP_K),
    dtype=np.bool_
)


# --------------------------------------------------------------------------
# VERIFY REQUIRED COLUMNS
# --------------------------------------------------------------------------

required_topk_columns = [
    "canonical_image_id",
    "object_name_clean",
    "confidence"
] + SPATIAL_COLUMNS

missing_columns = [
    c for c in required_topk_columns
    if c not in topk_df.columns
]

assert not missing_columns, (
    f"Missing columns in topk_df: {missing_columns}"
)


# --------------------------------------------------------------------------
# BUILD TENSORS
# --------------------------------------------------------------------------

grouped = topk_df.groupby(
    "canonical_image_id",
    sort=False
)


for image_id, group in tqdm(
    grouped,
    total=grouped.ngroups,
    desc="Building object tensors"
):

    if image_id not in image_to_index:
        continue

    image_index = image_to_index[image_id]

    # Ensure highest confidence comes first
    group = group.sort_values(
        "confidence",
        ascending=False
    ).reset_index(drop=True)

    # Only TOP_K
    group = group.head(TOP_K)

    for position, (_, row) in enumerate(group.iterrows()):

        object_name = str(
            row["object_name_clean"]
        ).strip().lower()

        object_id = object_vocabulary.get(
            object_name,
            object_vocabulary["<unk>"]
        )

        object_ids[
            image_index,
            position
        ] = int(object_id)

        spatial_features[
            image_index,
            position,
            :
        ] = np.asarray(
            row[SPATIAL_COLUMNS],
            dtype=np.float32
        )

        confidence_features[
            image_index,
            position
        ] = float(
            row["confidence"]
        )

        object_mask[
            image_index,
            position
        ] = True


# --------------------------------------------------------------------------
# REPORT
# --------------------------------------------------------------------------

print("\nTensor shapes:")

print(
    f"object_ids          : {object_ids.shape}"
)

print(
    f"spatial_features    : {spatial_features.shape}"
)

print(
    f"confidence_features : {confidence_features.shape}"
)

print(
    f"object_mask         : {object_mask.shape}"
)


# --------------------------------------------------------------------------
# COVERAGE
# --------------------------------------------------------------------------

images_with_objects = int(
    object_mask.any(axis=1).sum()
)

images_without_objects = (
    N_IMAGES -
    images_with_objects
)

object_slots_used = int(
    object_mask.sum()
)


print("\nObject coverage:")

print(
    f"Images with objects    : "
    f"{images_with_objects:,}"
)

print(
    f"Images without objects : "
    f"{images_without_objects:,}"
)

print(
    f"Object slots used      : "
    f"{object_slots_used:,}"
)

print(
    f"Maximum possible slots : "
    f"{N_IMAGES * TOP_K:,}"
)


# --------------------------------------------------------------------------
# FINAL SHAPE CHECKS
# --------------------------------------------------------------------------

assert object_ids.shape == (
    N_IMAGES,
    TOP_K
)

assert spatial_features.shape == (
    N_IMAGES,
    TOP_K,
    9
)

assert confidence_features.shape == (
    N_IMAGES,
    TOP_K
)

assert object_mask.shape == (
    N_IMAGES,
    TOP_K
)

assert images_with_objects <= N_IMAGES

print("\nCELL 12 COMPLETE.")

BUILDING FIXED-SIZE OBJECT TENSORS

Total images: 39874
TOP_K: 10


Building object tensors:   0%|          | 0/39864 [00:00<?, ?it/s]


Tensor shapes:
object_ids          : (39874, 10)
spatial_features    : (39874, 10, 9)
confidence_features : (39874, 10)
object_mask         : (39874, 10)

Object coverage:
Images with objects    : 39,864
Images without objects : 10
Object slots used      : 243,482
Maximum possible slots : 398,740

CELL 12 COMPLETE.


In [ ]:
# ============================================================================
# CELL 13 — OBJECT FEATURE VERIFICATION
# ============================================================================

print("=" * 70)
print("OBJECT FEATURE VERIFICATION")
print("=" * 70)


# --------------------------------------------------------------------------
# SHAPE VERIFICATION
# --------------------------------------------------------------------------

print("\nShapes:")

print("object_ids          :", object_ids.shape)
print("spatial_features    :", spatial_features.shape)
print("confidence_features :", confidence_features.shape)
print("object_mask         :", object_mask.shape)


# --------------------------------------------------------------------------
# NUMERICAL VERIFICATION
# --------------------------------------------------------------------------

assert np.isfinite(
    object_ids
).all()

assert np.isfinite(
    spatial_features
).all()

assert np.isfinite(
    confidence_features
).all()


# --------------------------------------------------------------------------
# SPATIAL RANGE
# --------------------------------------------------------------------------

spatial_min = float(
    spatial_features.min()
)

spatial_max = float(
    spatial_features.max()
)

print("\nSpatial feature range:")
print(f"Minimum: {spatial_min:.6f}")
print(f"Maximum: {spatial_max:.6f}")


assert spatial_min >= 0.0
assert spatial_max <= 1.0 + 1e-6


# --------------------------------------------------------------------------
# CONFIDENCE RANGE
# --------------------------------------------------------------------------

active_confidences = confidence_features[
    object_mask
]

if len(active_confidences) > 0:

    confidence_min = float(
        active_confidences.min()
    )

    confidence_max = float(
        active_confidences.max()
    )

else:

    confidence_min = 0.0
    confidence_max = 0.0


print("\nConfidence range:")
print(f"Minimum: {confidence_min:.6f}")
print(f"Maximum: {confidence_max:.6f}")


assert confidence_min >= CONF_THRESHOLD - 1e-6
assert confidence_max <= 1.0 + 1e-6


# --------------------------------------------------------------------------
# MASK VERIFICATION
# --------------------------------------------------------------------------

active_slots = int(
    object_mask.sum()
)

inactive_slots = int(
    (~object_mask).sum()
)

print("\nMask:")
print(f"Active slots  : {active_slots:,}")
print(f"Inactive slots: {inactive_slots:,}")


# --------------------------------------------------------------------------
# PADDED SLOT VERIFICATION
# --------------------------------------------------------------------------

# Inactive object slots should contain zeros.

inactive_object_values = object_ids[
    ~object_mask
]

inactive_spatial_values = spatial_features[
    ~object_mask
]

inactive_confidence_values = confidence_features[
    ~object_mask
]


assert np.all(
    inactive_object_values == 0
)

assert np.allclose(
    inactive_spatial_values,
    0.0
)

assert np.allclose(
    inactive_confidence_values,
    0.0
)


print("\nPadded slots correctly contain zeros.")


# --------------------------------------------------------------------------
# OBJECT ID RANGE
# --------------------------------------------------------------------------

max_object_id = int(
    object_ids.max()
)

print(
    f"\nMaximum object ID: {max_object_id}"
)

assert max_object_id < len(
    object_vocabulary
)


print("\n" + "=" * 70)
print("CELL 13 COMPLETE — OBJECT FEATURES VERIFIED")
print("=" * 70)

OBJECT FEATURE VERIFICATION

Shapes:
object_ids          : (39874, 10)
spatial_features    : (39874, 10, 9)
confidence_features : (39874, 10)
object_mask         : (39874, 10)

Spatial feature range:
Minimum: 0.000000
Maximum: 1.000000

Confidence range:
Minimum: 0.300002
Maximum: 0.986814

Mask:
Active slots  : 243,482
Inactive slots: 155,258

Padded slots correctly contain zeros.

Maximum object ID: 81

CELL 13 COMPLETE — OBJECT FEATURES VERIFIED


In [ ]:
# ============================================================================
# CELL 14 — IMAGE METADATA
# ============================================================================

print("=" * 70)
print("BUILDING IMAGE METADATA")
print("=" * 70)


# --------------------------------------------------------------------------
# BASE METADATA
# --------------------------------------------------------------------------

metadata_df = captions_df[
    [
        "image_id",
        "image_name",
        "image_path"
    ]
].drop_duplicates(
    subset=["image_id"]
).copy()


# --------------------------------------------------------------------------
# OBJECT STATISTICS
# --------------------------------------------------------------------------

object_counts = object_mask.sum(
    axis=1
).astype(np.int16)

has_objects = (
    object_counts > 0
)


metadata_df["object_count"] = (
    metadata_df["image_id"]
    .map(
        {
            image_id: int(
                object_counts[idx]
            )
            for idx, image_id
            in enumerate(all_image_ids)
        }
    )
)


metadata_df["has_objects"] = (
    metadata_df["object_count"] > 0
)


# --------------------------------------------------------------------------
# PRESERVE EXACT IMAGE ORDER
# --------------------------------------------------------------------------

metadata_df["image_index"] = (
    metadata_df["image_id"]
    .map(image_to_index)
)


metadata_df = metadata_df.sort_values(
    "image_index"
).reset_index(
    drop=True
)


# --------------------------------------------------------------------------
# CHECK
# --------------------------------------------------------------------------

print(
    f"\nMetadata rows: "
    f"{len(metadata_df):,}"
)

print(
    f"Images with objects: "
    f"{metadata_df['has_objects'].sum():,}"
)

print(
    f"Images without objects: "
    f"{(~metadata_df['has_objects']).sum():,}"
)


assert len(metadata_df) == N_IMAGES

assert metadata_df["image_id"].nunique() == N_IMAGES


print("\nCELL 14 COMPLETE.")

BUILDING IMAGE METADATA

Metadata rows: 39,874
Images with objects: 39,864
Images without objects: 10

CELL 14 COMPLETE.


In [ ]:
# ============================================================================
# CELL 15 — IMAGE-LEVEL TRAIN / VAL / TEST SPLIT
# ============================================================================

print("=" * 70)
print("IMAGE-LEVEL TRAIN / VAL / TEST SPLIT")
print("=" * 70)


# --------------------------------------------------------------------------
# REPRODUCIBLE SHUFFLE
# --------------------------------------------------------------------------

rng = np.random.default_rng(
    SEED
)

shuffled_ids = np.array(
    all_image_ids,
    dtype=object
)

rng.shuffle(
    shuffled_ids
)


# --------------------------------------------------------------------------
# SPLIT SIZES
# --------------------------------------------------------------------------

n_train = int(
    N_IMAGES * TRAIN_RATIO
)

n_val = int(
    N_IMAGES * VAL_RATIO
)

n_test = (
    N_IMAGES -
    n_train -
    n_val
)


# --------------------------------------------------------------------------
# SPLIT
# --------------------------------------------------------------------------

train_ids = shuffled_ids[
    :n_train
]

val_ids = shuffled_ids[
    n_train:
    n_train + n_val
]

test_ids = shuffled_ids[
    n_train + n_val:
]


# --------------------------------------------------------------------------
# REPORT
# --------------------------------------------------------------------------

print(
    f"\nTrain images: {len(train_ids):,}"
)

print(
    f"Validation images: {len(val_ids):,}"
)

print(
    f"Test images: {len(test_ids):,}"
)

print(
    f"Total: "
    f"{len(train_ids) + len(val_ids) + len(test_ids):,}"
)


# --------------------------------------------------------------------------
# NO LEAKAGE
# --------------------------------------------------------------------------

train_set = set(train_ids)
val_set = set(val_ids)
test_set = set(test_ids)


assert len(
    train_set & val_set
) == 0

assert len(
    train_set & test_set
) == 0

assert len(
    val_set & test_set
) == 0


assert (
    len(train_ids) +
    len(val_ids) +
    len(test_ids)
) == N_IMAGES


# --------------------------------------------------------------------------
# EXPECTED APPROXIMATE SPLIT
# --------------------------------------------------------------------------

assert len(train_ids) == 31_899
assert len(val_ids) == 3_987
assert len(test_ids) == 3_988


print("\nNo image-level leakage detected.")

print("\nCELL 15 COMPLETE.")

IMAGE-LEVEL TRAIN / VAL / TEST SPLIT

Train images: 31,899
Validation images: 3,987
Test images: 3,988
Total: 39,874

No image-level leakage detected.

CELL 15 COMPLETE.


In [ ]:
# ============================================================================
# CELL 16 — CAPTION VOCABULARY
# ============================================================================

print("=" * 70)
print("BUILDING CAPTION VOCABULARY")
print("=" * 70)


# --------------------------------------------------------------------------
# TRAINING IMAGE SET
# --------------------------------------------------------------------------

train_id_set = set(
    train_ids
)


train_captions = captions_df[
    captions_df["image_id"].isin(
        train_id_set
    )
].copy()


print(
    f"\nTraining images: "
    f"{len(train_id_set):,}"
)

print(
    f"Training captions: "
    f"{len(train_captions):,}"
)


# --------------------------------------------------------------------------
# TOKEN FREQUENCY
# --------------------------------------------------------------------------

word_counter = Counter()

for tokens in train_captions["tokens"]:

    if isinstance(tokens, (list, tuple)):

        word_counter.update(tokens)


# --------------------------------------------------------------------------
# SPECIAL TOKENS
# --------------------------------------------------------------------------

caption_vocabulary = {
    "<pad>": 0,
    "<start>": 1,
    "<end>": 2,
    "<unk>": 3
}


# --------------------------------------------------------------------------
# ADD WORDS
# --------------------------------------------------------------------------

for word, count in sorted(
    word_counter.items(),
    key=lambda x: (-x[1], x[0])
):

    if count >= MIN_WORD_FREQ:

        if word not in caption_vocabulary:

            caption_vocabulary[word] = (
                len(caption_vocabulary)
            )


# --------------------------------------------------------------------------
# REVERSE VOCABULARY
# --------------------------------------------------------------------------

index_to_word = {
    idx: word
    for word, idx
    in caption_vocabulary.items()
}


# --------------------------------------------------------------------------
# REPORT
# --------------------------------------------------------------------------

print(
    f"\nUnique words in training captions: "
    f"{len(word_counter):,}"
)

print(
    f"Vocabulary size: "
    f"{len(caption_vocabulary):,}"
)

print("\nSpecial tokens:")

print(
    "<pad>  =",
    caption_vocabulary["<pad>"]
)

print(
    "<start> =",
    caption_vocabulary["<start>"]
)

print(
    "<end>   =",
    caption_vocabulary["<end>"]
)

print(
    "<unk>   =",
    caption_vocabulary["<unk>"]
)


assert caption_vocabulary["<pad>"] == 0
assert caption_vocabulary["<start>"] == 1
assert caption_vocabulary["<end>"] == 2
assert caption_vocabulary["<unk>"] == 3


print("\nCELL 16 COMPLETE.")

BUILDING CAPTION VOCABULARY

Training images: 31,899
Training captions: 159,494

Unique words in training captions: 17,091
Vocabulary size: 12,174

Special tokens:
<pad>  = 0
<start> = 1
<end>   = 2
<unk>   = 3

CELL 16 COMPLETE.


In [ ]:
# ============================================================================
# CELL 17 — CAPTION SEQUENCE CREATION
# ============================================================================

print("=" * 70)
print("CREATING CAPTION SEQUENCES")
print("=" * 70)


# --------------------------------------------------------------------------
# TOKEN IDS
# --------------------------------------------------------------------------

PAD_ID = caption_vocabulary["<pad>"]
START_ID = caption_vocabulary["<start>"]
END_ID = caption_vocabulary["<end>"]
UNK_ID = caption_vocabulary["<unk>"]


# --------------------------------------------------------------------------
# ENCODER
# --------------------------------------------------------------------------

def encode_caption(tokens):

    sequence = [
        START_ID
    ]

    for token in tokens:

        sequence.append(
            caption_vocabulary.get(
                token,
                UNK_ID
            )
        )

    sequence.append(
        END_ID
    )

    # Keep MAX_LEN tokens
    sequence = sequence[
        :MAX_LEN
    ]

    # Ensure END token exists if truncation removed it
    if sequence[-1] != END_ID:

        sequence[-1] = END_ID


    # Padding
    if len(sequence) < MAX_LEN:

        sequence.extend(
            [PAD_ID] *
            (
                MAX_LEN -
                len(sequence)
            )
        )

    return sequence


# --------------------------------------------------------------------------
# CREATE SEQUENCES FOR ALL CAPTIONS
# --------------------------------------------------------------------------

caption_sequences = np.array(
    [
        encode_caption(tokens)
        for tokens in captions_df["tokens"]
    ],
    dtype=np.int32
)


# --------------------------------------------------------------------------
# REPORT
# --------------------------------------------------------------------------

print(
    f"\nCaption sequences shape: "
    f"{caption_sequences.shape}"
)

print(
    f"Expected rows: "
    f"{len(captions_df):,}"
)

print(
    f"Maximum sequence length: "
    f"{MAX_LEN}"
)


# --------------------------------------------------------------------------
# CHECK
# --------------------------------------------------------------------------

assert caption_sequences.shape == (
    len(captions_df),
    MAX_LEN
)

assert caption_sequences.dtype == np.int32

assert caption_sequences.min() >= 0

assert caption_sequences.max() < len(
    caption_vocabulary
)


print("\nCELL 17 COMPLETE.")

CREATING CAPTION SEQUENCES

Caption sequences shape: (199369, 32)
Expected rows: 199,369
Maximum sequence length: 32

CELL 17 COMPLETE.


In [ ]:
# ============================================================================
# CELL 18 — NLP DATASET SPLITS
# ============================================================================

print("=" * 70)
print("BUILDING NLP DATASET SPLITS")
print("=" * 70)


# --------------------------------------------------------------------------
# IMAGE → SPLIT
# --------------------------------------------------------------------------

image_split = {}

for image_id in train_ids:
    image_split[image_id] = "train"

for image_id in val_ids:
    image_split[image_id] = "val"

for image_id in test_ids:
    image_split[image_id] = "test"


# --------------------------------------------------------------------------
# CAPTION SPLIT
# --------------------------------------------------------------------------

caption_split = captions_df[
    "image_id"
].map(
    image_split
)


# --------------------------------------------------------------------------
# CHECK UNKNOWN
# --------------------------------------------------------------------------

unknown_split_count = (
    caption_split.isna().sum()
)

assert unknown_split_count == 0, (
    f"{unknown_split_count} captions "
    f"could not be assigned to a split."
)


# --------------------------------------------------------------------------
# BOOLEAN INDICES
# --------------------------------------------------------------------------

train_caption_mask = (
    caption_split == "train"
).to_numpy()

val_caption_mask = (
    caption_split == "val"
).to_numpy()

test_caption_mask = (
    caption_split == "test"
).to_numpy()


# --------------------------------------------------------------------------
# INDICES
# --------------------------------------------------------------------------

train_caption_indices = np.where(
    train_caption_mask
)[0]

val_caption_indices = np.where(
    val_caption_mask
)[0]

test_caption_indices = np.where(
    test_caption_mask
)[0]


# --------------------------------------------------------------------------
# REPORT
# --------------------------------------------------------------------------

print(
    f"\nTrain captions: "
    f"{len(train_caption_indices):,}"
)

print(
    f"Validation captions: "
    f"{len(val_caption_indices):,}"
)

print(
    f"Test captions: "
    f"{len(test_caption_indices):,}"
)

print(
    f"Total captions: "
    f"{len(captions_df):,}"
)


# --------------------------------------------------------------------------
# EXPECTED COUNTS
# --------------------------------------------------------------------------

assert (
    len(train_caption_indices) +
    len(val_caption_indices) +
    len(test_caption_indices)
) == len(captions_df)


# --------------------------------------------------------------------------
# VERIFY IMAGE-LEVEL CONSISTENCY
# --------------------------------------------------------------------------

assert (
    set(
        captions_df.iloc[
            train_caption_indices
        ]["image_id"]
    )
    <= train_set
)

assert (
    set(
        captions_df.iloc[
            val_caption_indices
        ]["image_id"]
    )
    <= val_set
)

assert (
    set(
        captions_df.iloc[
            test_caption_indices
        ]["image_id"]
    )
    <= test_set
)


print(
    "\nCaption splits correctly follow "
    "the image-level split."
)

print("\nCELL 18 COMPLETE.")

BUILDING NLP DATASET SPLITS

Train captions: 159,494
Validation captions: 19,935
Test captions: 19,940
Total captions: 199,369

Caption splits correctly follow the image-level split.

CELL 18 COMPLETE.


In [ ]:
# ============================================================================
# CELL 19 — SAVE OBJECT FEATURES
# ============================================================================

print("=" * 70)
print("SAVING OBJECT FEATURES")
print("=" * 70)


OBJECT_FEATURE_FILE = (
    OUTPUT_DIR /
    "rtdetr_structured_object_features.npz"
)


np.savez_compressed(
    OBJECT_FEATURE_FILE,

    image_ids=np.array(
        all_image_ids,
        dtype=object
    ),

    object_ids=object_ids,

    spatial_features=spatial_features,

    confidence_features=confidence_features,

    object_mask=object_mask
)


# --------------------------------------------------------------------------
# SAVE OBJECT VOCABULARY
# --------------------------------------------------------------------------

OBJECT_VOCAB_FILE = (
    OUTPUT_DIR /
    "object_vocabulary.json"
)


with open(
    OBJECT_VOCAB_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        object_vocabulary,
        f,
        indent=2,
        ensure_ascii=False
    )


# --------------------------------------------------------------------------
# REPORT
# --------------------------------------------------------------------------

print(
    f"\nObject feature file:"
)

print(
    OBJECT_FEATURE_FILE
)

print(
    f"Size: "
    f"{OBJECT_FEATURE_FILE.stat().st_size / (1024**2):.2f} MB"
)

print(
    "\nObject vocabulary:"
)

print(
    OBJECT_VOCAB_FILE
)

print(
    f"Vocabulary size: "
    f"{len(object_vocabulary):,}"
)


assert OBJECT_FEATURE_FILE.exists()
assert OBJECT_VOCAB_FILE.exists()


print("\nCELL 19 COMPLETE.")

SAVING OBJECT FEATURES

Object feature file:
/content/drive/MyDrive/Image_captioning/Outputs/Structured_Captioning/rtdetr_structured_object_features.npz
Size: 8.89 MB

Object vocabulary:
/content/drive/MyDrive/Image_captioning/Outputs/Structured_Captioning/object_vocabulary.json
Vocabulary size: 82

CELL 19 COMPLETE.


In [ ]:
# ============================================================================
# CELL 20 — SAVE CAPTION DATA
# ============================================================================

print("=" * 70)
print("SAVING CAPTION DATA")
print("=" * 70)


CAPTION_SEQUENCE_FILE = (
    OUTPUT_DIR /
    "caption_sequences.npz"
)


# --------------------------------------------------------------------------
# SAVE SEQUENCES
# --------------------------------------------------------------------------

np.savez_compressed(
    CAPTION_SEQUENCE_FILE,

    image_ids=np.array(
        captions_df["image_id"],
        dtype=object
    ),

    sequences=caption_sequences,

    train_indices=train_caption_indices,

    val_indices=val_caption_indices,

    test_indices=test_caption_indices
)


# --------------------------------------------------------------------------
# SAVE CAPTION VOCABULARY
# --------------------------------------------------------------------------

CAPTION_VOCAB_FILE = (
    OUTPUT_DIR /
    "caption_vocabulary.json"
)


with open(
    CAPTION_VOCAB_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        caption_vocabulary,
        f,
        indent=2,
        ensure_ascii=False
    )


# --------------------------------------------------------------------------
# SAVE CAPTION METADATA
# --------------------------------------------------------------------------

caption_metadata = captions_df[
    [
        "image_id",
        "image_name",
        "caption"
    ]
].copy()


caption_metadata["split"] = (
    caption_split.values
)


CAPTION_METADATA_FILE = (
    OUTPUT_DIR /
    "caption_metadata.csv"
)


caption_metadata.to_csv(
    CAPTION_METADATA_FILE,
    index=False
)


# --------------------------------------------------------------------------
# REPORT
# --------------------------------------------------------------------------

print(
    f"\nCaption sequence file:"
)

print(
    CAPTION_SEQUENCE_FILE
)

print(
    f"Size: "
    f"{CAPTION_SEQUENCE_FILE.stat().st_size / (1024**2):.2f} MB"
)

print(
    f"\nCaption vocabulary:"
)

print(
    CAPTION_VOCAB_FILE
)

print(
    f"\nCaption metadata:"
)

print(
    CAPTION_METADATA_FILE
)


assert CAPTION_SEQUENCE_FILE.exists()
assert CAPTION_VOCAB_FILE.exists()
assert CAPTION_METADATA_FILE.exists()


print("\nCELL 20 COMPLETE.")

SAVING CAPTION DATA

Caption sequence file:
/content/drive/MyDrive/Image_captioning/Outputs/Structured_Captioning/caption_sequences.npz
Size: 4.26 MB

Caption vocabulary:
/content/drive/MyDrive/Image_captioning/Outputs/Structured_Captioning/caption_vocabulary.json

Caption metadata:
/content/drive/MyDrive/Image_captioning/Outputs/Structured_Captioning/caption_metadata.csv

CELL 20 COMPLETE.


In [ ]:
# ============================================================================
# CELL 21 — SAVE PIPELINE CONFIGURATION
# ============================================================================

print("=" * 70)
print("SAVING PIPELINE CONFIGURATION")
print("=" * 70)


CONFIG_FILE = (
    OUTPUT_DIR /
    "structured_captioning_config.json"
)


config = {

    # ----------------------------------------------------------------------
    # DATASET
    # ----------------------------------------------------------------------

    "dataset": [
        "Flickr8K",
        "Flickr30K"
    ],

    "total_images": int(
        N_IMAGES
    ),

    "total_captions": int(
        len(captions_df)
    ),


    # ----------------------------------------------------------------------
    # DETECTOR
    # ----------------------------------------------------------------------

    "detector": "RT-DETR-X",

    "confidence_threshold": float(
        CONF_THRESHOLD
    ),

    "top_k": int(
        TOP_K
    ),


    # ----------------------------------------------------------------------
    # CAPTION
    # ----------------------------------------------------------------------

    "max_caption_length": int(
        MAX_LEN
    ),

    "minimum_word_frequency": int(
        MIN_WORD_FREQ
    ),


    # ----------------------------------------------------------------------
    # SPLIT
    # ----------------------------------------------------------------------

    "train_ratio": float(
        TRAIN_RATIO
    ),

    "validation_ratio": float(
        VAL_RATIO
    ),

    "test_ratio": float(
        TEST_RATIO
    ),

    "random_seed": int(
        SEED
    ),


    # ----------------------------------------------------------------------
    # SPATIAL FEATURES
    # ----------------------------------------------------------------------

    "spatial_feature_dimensions": 9,

    "spatial_features": [
        "xmin",
        "ymin",
        "xmax",
        "ymax",
        "center_x",
        "center_y",
        "box_width",
        "box_height",
        "box_area"
    ],


    # ----------------------------------------------------------------------
    # FEATURE STRUCTURE
    # ----------------------------------------------------------------------

    "object_feature_dimensions": {

        "object_id": 1,

        "spatial": 9,

        "confidence": 1

    },


    # ----------------------------------------------------------------------
    # TENSOR SHAPES
    # ----------------------------------------------------------------------

    "tensor_shapes": {

        "object_ids": list(
            object_ids.shape
        ),

        "spatial_features": list(
            spatial_features.shape
        ),

        "confidence_features": list(
            confidence_features.shape
        ),

        "object_mask": list(
            object_mask.shape
        ),

        "caption_sequences": list(
            caption_sequences.shape
        )

    },


    # ----------------------------------------------------------------------
    # VOCABULARIES
    # ----------------------------------------------------------------------

    "object_vocabulary_size": int(
        len(object_vocabulary)
    ),

    "caption_vocabulary_size": int(
        len(caption_vocabulary)
    ),


    # ----------------------------------------------------------------------
    # SPECIAL TOKENS
    # ----------------------------------------------------------------------

    "caption_special_tokens": {

        "pad": int(PAD_ID),

        "start": int(START_ID),

        "end": int(END_ID),

        "unk": int(UNK_ID)

    },

    "object_special_tokens": {

        "pad": int(
            object_vocabulary["<pad>"]
        ),

        "unk": int(
            object_vocabulary["<unk>"]
        )

    }

}


with open(
    CONFIG_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        config,
        f,
        indent=2
    )


print(
    f"\nConfiguration saved:"
)

print(
    CONFIG_FILE
)

print(
    f"Size: "
    f"{CONFIG_FILE.stat().st_size / 1024:.2f} KB"
)


assert CONFIG_FILE.exists()


print("\nCELL 21 COMPLETE.")

SAVING PIPELINE CONFIGURATION

Configuration saved:
/content/drive/MyDrive/Image_captioning/Outputs/Structured_Captioning/structured_captioning_config.json
Size: 1.12 KB

CELL 21 COMPLETE.


In [ ]:
# ============================================================================
# CELL 22 — FINAL VERIFICATION
# ============================================================================

print("=" * 70)
print("FINAL VERIFICATION — NOTEBOOK 10")
print("=" * 70)


# ============================================================================
# DATASET
# ============================================================================

print("\nDATASET")
print("-" * 70)

print(
    f"Images          : {N_IMAGES:,}"
)

print(
    f"Captions        : {len(captions_df):,}"
)


assert N_IMAGES == 39_874

assert len(captions_df) == 199_369

assert captions_df["image_id"].nunique() == 39_874


# ============================================================================
# OBJECT TENSORS
# ============================================================================

print("\nOBJECT TENSORS")
print("-" * 70)

print(
    f"Object IDs          : {object_ids.shape}"
)

print(
    f"Spatial features    : {spatial_features.shape}"
)

print(
    f"Confidence features : {confidence_features.shape}"
)

print(
    f"Object mask         : {object_mask.shape}"
)


assert object_ids.shape == (
    39_874,
    TOP_K
)

assert spatial_features.shape == (
    39_874,
    TOP_K,
    9
)

assert confidence_features.shape == (
    39_874,
    TOP_K
)

assert object_mask.shape == (
    39_874,
    TOP_K
)


# ============================================================================
# CAPTION TENSORS
# ============================================================================

print("\nCAPTION TENSORS")
print("-" * 70)

print(
    f"Caption sequences: "
    f"{caption_sequences.shape}"
)


assert caption_sequences.shape == (
    199_369,
    MAX_LEN
)


# ============================================================================
# SPLITS
# ============================================================================

print("\nIMAGE SPLITS")
print("-" * 70)

print(
    f"Train: {len(train_ids):,}"
)

print(
    f"Val  : {len(val_ids):,}"
)

print(
    f"Test : {len(test_ids):,}"
)

assert len(train_ids) == 31_899
assert len(val_ids) == 3_987
assert len(test_ids) == 3_988


# ============================================================================
# LEAKAGE
# ============================================================================

assert len(
    train_set & val_set
) == 0

assert len(
    train_set & test_set
) == 0

assert len(
    val_set & test_set
) == 0


# ============================================================================
# FILES
# ============================================================================

print("\nOUTPUT FILES")
print("-" * 70)

expected_files = [

    "rtdetr_structured_object_features.npz",

    "object_vocabulary.json",

    "caption_sequences.npz",

    "caption_vocabulary.json",

    "caption_metadata.csv",

    "structured_captioning_config.json"

]


for filename in expected_files:

    path = OUTPUT_DIR / filename

    exists = path.exists()

    print(
        f"{filename:45s} "
        f"{'OK' if exists else 'MISSING'}"
    )

    assert exists


# ============================================================================
# NUMERICAL INTEGRITY
# ============================================================================

print("\nNUMERICAL INTEGRITY")
print("-" * 70)


assert np.isfinite(
    spatial_features
).all()

assert np.isfinite(
    confidence_features
).all()

assert np.isfinite(
    caption_sequences
).all()


assert spatial_features.min() >= 0.0

assert spatial_features.max() <= 1.0 + 1e-6


active_conf = confidence_features[
    object_mask
]

if len(active_conf) > 0:

    assert active_conf.min() >= (
        CONF_THRESHOLD - 1e-6
    )

    assert active_conf.max() <= (
        1.0 + 1e-6
    )


print("Numerical checks passed.")


# ============================================================================
# ZERO-DETECTION IMAGES
# ============================================================================

zero_detection_images = int(
    (~object_mask.any(axis=1)).sum()
)

print(
    f"\nImages without surviving objects: "
    f"{zero_detection_images}"
)

# These images remain in the dataset.
assert (
    zero_detection_images >= 0
)

assert (
    object_mask.shape[0] == 39_874
)


# ============================================================================
# FINAL STATUS
# ============================================================================

print("\n" + "=" * 70)
print("NOTEBOOK 10 COMPLETE")
print("=" * 70)

print(
    "\nRT-DETR-X structured object representation: READY"
)

print(
    "Caption sequences: READY"
)

print(
    "Image-level train/val/test split: READY"
)

print(
    "Saved features: READY"
)

print(
    "Ready for Notebook 11 — Object-Aware Transformer"
)

print("=" * 70)

FINAL VERIFICATION — NOTEBOOK 10

DATASET
----------------------------------------------------------------------
Images          : 39,874
Captions        : 199,369

OBJECT TENSORS
----------------------------------------------------------------------
Object IDs          : (39874, 10)
Spatial features    : (39874, 10, 9)
Confidence features : (39874, 10)
Object mask         : (39874, 10)

CAPTION TENSORS
----------------------------------------------------------------------
Caption sequences: (199369, 32)

IMAGE SPLITS
----------------------------------------------------------------------
Train: 31,899
Val  : 3,987
Test : 3,988

OUTPUT FILES
----------------------------------------------------------------------
rtdetr_structured_object_features.npz         OK
object_vocabulary.json                        OK
caption_sequences.npz                         OK
caption_vocabulary.json                       OK
caption_metadata.csv                          OK
structured_captioning_config.json  